In [0]:
%sql
---SHOW SCHEMAS IN retail;

--SELECT current_catalog(), current_schema();

In [0]:
%sql
--USE CATALOG retail;

--SELECT current_catalog(), current_schema();
--USE SCHEMA  dev_bronze;
--SELECT current_catalog(), current_schema();

In [0]:
%sql
--SHOW Schemas IN retail;
--SHOW TABLES IN retail.test_bronze;


## Datenimport mit PySpark

Der untere Code importiert die benötigten Bibliotheken und definiert den Quellpfad (`source`) für die Retail-Org-Datensätze.

Anschließend werden verschiedene Datensätze aus unterschiedlichen Dateiformaten eingelesen und jeweils als PySpark DataFrames gespeichert.

- **CSV:** Customers, Company Employees, Loyalty Segments, Products, Promotions und Suppliers.
- **Parquet:** Active Promotions.
- **JSON:** Sales Orders und Sales Stream. Durch `*.json` werden alle JSON-Dateien im jeweiligen Verzeichnis eingelesen.
- **XML:** Purchase Orders. Mit `rowTag="purchase_item"` wird festgelegt, welches XML-Element als einzelner Datensatz interpretiert wird.

Bei den CSV-Dateien wird `header=True` verwendet, damit die erste Zeile als Spaltenüberschrift erkannt wird. Für die Products-Datei wird zusätzlich `sep=";"` definiert, da die Werte durch Semikolons getrennt sind.

Zum Schluss werden mit `display(purchase_orders.limit(5))` die ersten fünf Datensätze der eingelesenen Purchase-Orders-Tabelle angezeigt.

In [0]:
dbutils.widgets.text("retail_env", "")

ENV = dbutils.widgets.get("retail_env").strip().lower()

if ENV not in ["dev", "test", "prod"]:
    raise ValueError(f"Invalid or missing environment: {ENV}")

CATALOG = "retail"

BRONZE_SCHEMA = f"{ENV}_bronze"
SILVER_SCHEMA = f"{ENV}_silver"
GOLD_SCHEMA = f"{ENV}_gold"

print(f"Current environment: {ENV}")
print(f"Catalog: {CATALOG}")
print(f"Bronze: {BRONZE_SCHEMA}")
print(f"Silver: {SILVER_SCHEMA}")
print(f"Gold: {GOLD_SCHEMA}")

Current environment: dev
Catalog: retail
Bronze: dev_bronze
Silver: dev_silver
Gold: dev_gold


In [0]:
from pyspark.sql import functions as F
import os


source="/databricks-datasets/retail-org"
customers = spark.read.csv(f"{source}/customers/customers.csv", header=True, sep=",")
company_employees=spark.read.csv(f"{source}/company_employees/company_employees.csv",  header=True, sep=",")
loyalty_segments=spark.read.csv(f"{source}/loyalty_segments/loyalty_segment.csv", header=True,sep=",")
products=spark.read.csv(f"{source}/products/products.csv", sep=";",  header=True)
promotions=spark.read.csv(f"{source}/promotions/promotions.csv", sep=",",  header=True)
suppliers=spark.read.csv(f"{source}/suppliers/suppliers.csv", sep=",",  header=True)
active_promotions = spark.read.parquet(f"{source}/active_promotions/active_promotions.parquet")

sales_orders = spark.read.json(f"{source}/sales_orders/*.json")

sales_stream = spark.read.json(f"{source}/sales_stream/*.json")


purchase_orders = (
    spark.read
    .format("xml")
    .option("rowTag", "purchase_item")
    .load(f"{source}/purchase_orders/purchase_orders.xml")
)



## Erstellung des Bronze-Schemas und Speicherung der Daten als Delta Tables

Der untere Code erstellt zunächst das Schema `Retailorg_bronze`, sofern dieses noch nicht existiert.

Anschließend werden die zuvor eingelesenen DataFrames in einem Python-Dictionary (`tables`) zusammengefasst. Dabei wird jedem DataFrame ein entsprechender Tabellenname zugeordnet.

Mit einer `for`-Schleife werden alle DataFrames iteriert und als **Delta Tables** in der Datenbank `retail.dev_bronze` gespeichert.

Dabei werden folgende Optionen verwendet:

- `format("delta")`: Speichert die Daten im Delta-Format.
- `mode("overwrite")`: Überschreibt eine bereits vorhandene Tabelle vollständig.
- `saveAsTable()`: Speichert den DataFrame als persistente Tabelle im Metastore.

Dadurch werden die eingelesenen Quelldaten als Delta Tables in der Bronze-Schicht abgelegt.

In [0]:
# ==========================================
# BRONZE LAYER - AUTOMATIC TABLE INGESTION
# ==========================================

# Create environment schema if not exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")

# Register all source DataFrames
bronze_tables = {
    "customers": customers,
    "company_employees": company_employees,
    "loyalty_segments": loyalty_segments,
    "products": products,
    "promotions": promotions,
    "suppliers": suppliers,
    "active_promotions": active_promotions,
    "sales_orders": sales_orders,
    "sales_stream": sales_stream,
    "purchase_orders": purchase_orders
}

# Automatically write all tables
for table_name, df in bronze_tables.items():

    target_table = f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    print(f"SUCCESS: {target_table}")

print("===================================")
print(f"BRONZE INGESTION COMPLETED: {ENV}")
print("===================================")



SUCCESS: retail.dev_bronze.customers
SUCCESS: retail.dev_bronze.company_employees
SUCCESS: retail.dev_bronze.loyalty_segments
SUCCESS: retail.dev_bronze.products
SUCCESS: retail.dev_bronze.promotions
SUCCESS: retail.dev_bronze.suppliers
SUCCESS: retail.dev_bronze.active_promotions
SUCCESS: retail.dev_bronze.sales_orders
SUCCESS: retail.dev_bronze.sales_stream
SUCCESS: retail.dev_bronze.purchase_orders
BRONZE INGESTION COMPLETED: dev


In [0]:
table_n='customers'

customers=spark.read.table(f"retail.dev_bronze.{table_n}")

customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- tax_id: string (nullable = true)
 |-- tax_code: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- state: string (nullable = true)
 |-- city: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- street: string (nullable = true)
 |-- number: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- region: string (nullable = true)
 |-- district: string (nullable = true)
 |-- lon: string (nullable = true)
 |-- lat: string (nullable = true)
 |-- ship_to_address: string (nullable = true)
 |-- valid_from: string (nullable = true)
 |-- valid_to: string (nullable = true)
 |-- units_purchased: string (nullable = true)
 |-- loyalty_segment: string (nullable = true)



In [0]:
display(customers.limit(2))

customer_id,tax_id,tax_code,customer_name,state,city,postcode,street,number,unit,region,district,lon,lat,ship_to_address,valid_from,valid_to,units_purchased,loyalty_segment
11123757,null,null,"SMITH, SHIRLEY",IN,BREMEN,46506.0,N CENTER ST,521.0,null,Indiana,50.0,-86.1465825,41.4507625,"IN, 46506.0, N CENTER ST, 521.0",1532824233,1548137353.0,34.0,3
30585978,null,null,"STEPHENS, GERALDINE M",OR,ADDRESS,0,NO SITUS,null,null,null,null,-122.1055158,45.374317,"OR, 0, NO SITUS, nan",1523100473,null,18.0,3


In [0]:
active_promotions.printSchema()

root
 |-- promo_customer: string (nullable = true)
 |-- promo_item: string (nullable = true)
 |-- promo_disc: double (nullable = true)
 |-- promo_id: long (nullable = true)
 |-- promo_datetime: string (nullable = true)
 |-- promo_qty: long (nullable = true)
 |-- cumsum: long (nullable = true)
 |-- promo_began: string (nullable = true)
 |-- units_required: long (nullable = true)
 |-- eligible: long (nullable = true)
 |-- deadline: string (nullable = true)

